# 电力看经济智能体：完整流程 Notebook

本 Notebook 依次演示数据获取、特征构建、模型加载/训练、异常扫描、诊断、报告和问答。默认使用 `configs/demo.yaml`。第一次运行前请在项目根目录执行 `pip install -e .`。

In [ ]:
import json
from pathlib import Path

import pandas as pd

from power_econ.agents import PowerEconomyOrchestrator
from power_econ.config import load_settings
from power_econ.data import collect_raw_data
from power_econ.features.builder import FeatureBuilder
from power_econ.models import TrainingPipeline

CONFIG = "../configs/demo.yaml" if Path.cwd().name == "notebooks" else "configs/demo.yaml"
settings = load_settings(CONFIG)
settings

## 1. 获取并校验多源数据

In [ ]:
raw = collect_raw_data(settings)
print(raw.shape)
raw[["timestamp", "load_mw", "temperature_2m", "pmi", "policy_event"]].head()

## 2. 构建因果特征和五个特征组

In [ ]:
features, spec = FeatureBuilder(settings).run(raw)
print(f"rows={len(features):,}, features={len(spec.feature_columns)}, groups={list(spec.groups)}")
pd.Series({k: len(v) for k, v in spec.groups.items()}, name="feature_count")

## 3. 训练模型（可选）

仓库已附带训练产物。需要从头训练时，把 `RUN_TRAIN` 改为 `True`。

In [ ]:
RUN_TRAIN = False
if RUN_TRAIN:
    summary = TrainingPipeline(settings).run(features, spec)
    print(json.dumps(summary.model_dump(mode="json"), ensure_ascii=False, indent=2))
else:
    summary_path = settings.resolve(settings.paths.output_dir / "training_summary.json")
    summary = json.load(open(summary_path, encoding="utf-8"))
    print(json.dumps(summary["forecaster_metrics"], ensure_ascii=False, indent=2))

## 4. 加载编排器并输出最新预测

In [ ]:
orch = PowerEconomyOrchestrator.from_settings(settings)
forecast = orch.perception.latest_forecast()
forecast.head()

## 5. 扫描异常

In [ ]:
events = sorted(
    orch.monitor("2024-05-01", "2024-05-10"),
    key=lambda e: e.anomaly_score,
    reverse=True,
)
pd.DataFrame([e.model_dump(mode="json") for e in events[:10]])

## 6. 诊断最高分事件

In [ ]:
if events:
    diagnosis = orch.diagnosis.diagnose(events[0])
    print(diagnosis.narrative)
    display(pd.DataFrame([x.model_dump() for x in diagnosis.top_causes]))
    display(pd.DataFrame([x.model_dump() for x in diagnosis.feature_contributions]))
else:
    print("指定时间段没有超过阈值的代表性异常。")

## 7. 生成报告

In [ ]:
report = orch.generate_report("2024-05-01", "2024-05-10", max_diagnoses=5)
print(orch.report.to_markdown(report))

## 8. 知识增强问答

In [ ]:
answer = orch.ask("这段时间的异常能否直接说明经济走弱？", session_id="notebook")
print(answer.answer)
print("\n证据：")
for item in answer.evidence:
    print("-", item)

## 9. 下一步

把 `configs/real_data.yaml` 或 `configs/csv_data.yaml` 复制为本地区配置，按 `docs/真实数据接入规范.md` 准备文件，然后重复同一流程。